# {Project Title}📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
DATA_PATH = "data/your_data.csv"     # path to your CSV file
TARGET_COL = "target_column_name"    # name of the column you want to predict


## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
📝 <!-- Answer Below -->

**“How do weather conditions (temperature and precipitation) and calendar factors (weekdays, weekends, and holidays) affect daily bike‑share usage in New York City?”**

To make this concrete, I will look at:
- How total daily trips change with temperature.
- How much ridership drops on rainy or snowy days compared to clear days.
- How weekends and holidays differ from regular weekdays in terms of trip volume.


## What would an answer look like?
*What is your hypothesized answer to your question?*
📝 <!-- Answer Below -->

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
📝 <!-- Answer Below -->
### **1. Bike‑Share Trip Data (CSV File)**
This dataset contains detailed bike‑share trip information, including:
- start and end times
- station IDs
- user types
- ride duration  
- latitude/longitude (optional)

I will aggregate this dataset to **daily total trips** so I can analyze demand per day.  
This provides my **target variable**: `total_trips`.

---

### **2. Historical Weather Data (API)**
I will use an open weather API (such as **Open‑Meteo Historical Weather API**) to retrieve:
- daily average temperature  
- daily total precipitation  
- weather type (clear, rain, snow, etc.)

This dataset contains the external environmental conditions that might explain changes in bike‑share usage.

---

### **3. Holiday & Calendar Data (CSV File or API)**
This dataset provides:
- U.S. federal holidays
- names of each holiday
- dates of each event

From this I will create features such as:
- `is_holiday`
- `is_weekend`

---

### **How They Are Related**
All three datasets share a common link: **date**.

I will:
1. Aggregate bike‑share trips by **date** → daily total trips  
2. Aggregate weather data by **date** → avg_temp, total_precip, weather_type  
3. Merge holiday/calendar data by **date** → is_holiday  

After merging on the `date` column, I will produce one final dataset with rows like:

| date | total_trips | avg_temp | total_precip | weather_type | is_weekend | is_holiday |
|------|--------------|-----------|---------------|--------------|-------------|-------------|

This unified dataset will allow me to perform EDA and build machine learning models that predict bike‑share usage based on weather and calendar features.


## Approach and Analysis
*What is your approach to answering your project question?  
How will you use the identified data to answer your project question?*

📝 My approach combines exploratory data analysis (EDA) and machine learning to understand how weather and calendar factors influence bike‑share usage. The overall workflow I will follow is the **Ask → Prepare → Process → Analyze → Evaluate → Share** cycle that we learned in class.

### **1. Ask**
The goal is to determine:
- How temperature and precipitation affect daily bike‑share usage.
- How weekends and holidays differ from weekdays.
- Whether machine learning models can predict daily total trips based on weather and calendar inputs.

### **2. Prepare**
I will load and merge three datasets:
1. Daily bike‑share trip counts  
2. Daily weather metrics (temperature, precipitation, weather type)  
3. Calendar/holiday information  

These datasets will be merged on the shared `date` column.  
After merging, I will inspect:
- Missing values  
- Outliers  
- Data types  
- Summary statistics  
- Initial patterns through plots  

### **3. Process**
I will create a preprocessing pipeline using **scikit‑learn** that automatically handles:
- Missing numeric values (median imputation)
- Missing categorical values (most frequent imputation)
- Categorical encoding (One‑Hot Encoding)
- Numerical scaling (StandardScaler)

This ensures the data is cleaned and transformed consistently during training and testing.

### **4. Analyze**
I will split the data into **80% training** and **20% testing** using `train_test_split`.

Then I will train multiple regression models:
- Linear Regression  
- Random Forest Regressor  
- KNN Regressor  

For each model, I will evaluate performance using:
- **R² score**
- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Square Error)**

This allows me to compare algorithms and choose the most effective one.

### **5. Evaluate**
After selecting the best model, I will:
- Inspect prediction accuracy on the test set  
- Plot true vs predicted values  
- Plot residuals to evaluate errors  

This will show how well weather and calendar variables explain daily ridership.

### **6. Share**
I will summarize:
- Which features had the strongest influence  
- How different weather conditions change usage  
- Which model performed best  
- Practical insights (e.g., “ridership drops X% on rainy days”)  

This connects the results back to stakeholders like city planners or bike‑share operators.

Below this markdown section, I will include all Python code needed to perform the above steps.


In [ ]:
# =======================
# CONFIG: ADJUST IF NEEDED
# =======================
ZIP_PATH = "2013-citibike-tripdata.zip"  # Name/path of your zip file in the repo
CSV_IN_ZIP = "2013-citibike-tripdata/201307-citibike-tripdata.csv"  # July 2013 trips
SAMPLE_SIZE = 50000  # Number of trips to sample for faster training (you can change this)
TARGET_COL = "tripduration"  # We will predict trip duration (in seconds)

# =======================
# IMPORTS
# =======================
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =======================
# 1) LOAD DATA FROM ZIP
# =======================
print("Loading Citi Bike data from zip file...")
with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(CSV_IN_ZIP) as f:
        df = pd.read_csv(f)

print("Loaded shape:", df.shape)
display(df.head())

# =======================
# 2) BASIC EDA
# =======================
print("\nData info:")
print(df.info())

print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

print("\nSummary statistics (numeric):")
display(df.describe())

# Plot distribution of tripduration (target)
plt.figure(figsize=(6, 4))
sns.histplot(df["tripduration"], kde=True)
plt.title("Distribution of tripduration (seconds)")
plt.xlabel("tripduration")
plt.tight_layout()
plt.show()

# =======================
# 3) FEATURE ENGINEERING
#    (time-based features)
# =======================
# Convert starttime to datetime
df["starttime"] = pd.to_datetime(df["starttime"])

# Derive hour of day, day of week, weekend flag
df["start_hour"] = df["starttime"].dt.hour
df["start_dayofweek"] = df["starttime"].dt.dayofweek  # 0=Mon, 6=Sun
df["is_weekend"] = df["start_dayofweek"] >= 5

# Optional: remove trips with extremely long durations (e.g. > 2 hours)
max_duration = 2 * 60 * 60  # 2 hours in seconds
df = df[df["tripduration"] > 0]
df = df[df["tripduration"] <= max_duration]

print("\nShape after filtering unrealistic durations:", df.shape)

# Sample a subset for faster training
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
    print(f"Sampled down to {SAMPLE_SIZE} rows for modeling.")

# =======================
# 4) DEFINE FEATURES & TARGET
# =======================
TARGET_COL = "tripduration"

feature_cols = [
    "start_hour",
    "start_dayofweek",
    "is_weekend",
    "start station id",
    "end station id",
    "usertype",
    "gender"
]

# Keep only selected columns
df_model = df[feature_cols + [TARGET_COL]].copy()

print("\nColumns used for modeling:")
print(df_model.columns)

X = df_model[feature_cols]
y = df_model[TARGET_COL]

# Identify numeric & categorical features
numeric_features = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["int64", "float64", "bool"]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"\nTrain size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")

# =======================
# 5) PREPROCESSING PIPELINE
# =======================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =======================
# 6) MODELS TO TEST
# =======================
models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "KNNRegressor": KNeighborsRegressor(n_neighbors=5)
}

results = []

for name, model in models.items():
    reg = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    
    results.append({
        "model": name,
        "r2": r2,
        "mae": mae,
        "rmse": rmse
    })
    
    print(f"\n=== {name} ===")
    print(f"R²:   {r2:.4f}")
    print(f"MAE:  {mae:.2f} seconds")
    print(f"RMSE: {rmse:.2f} seconds")
    print("-" * 40)

results_df = pd.DataFrame(results)
print("\nSummary of model performance:")
display(results_df.sort_values("rmse"))

# =======================
# 7) BEST MODEL & DIAGNOSTICS
# =======================
best_model_name = results_df.sort_values("rmse").iloc[0]["model"]
print(f"\nBest model based on RMSE: {best_model_name}")

best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[best_model_name])
])

best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)

# True vs predicted
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred_best, alpha=0.4)
plt.xlabel("True tripduration (seconds)")
plt.ylabel("Predicted tripduration (seconds)")
plt.title(f"True vs Predicted ({best_model_name})")

min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")

plt.tight_layout()
plt.show()

# Residuals
residuals = y_test - y_pred_best
plt.figure(figsize=(5, 4))
sns.histplot(residuals, kde=True)
plt.title("Residuals (true - predicted tripduration)")
plt.xlabel("Residual (seconds)")
plt.tight_layout()
plt.show()


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

In [ ]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

[NbConvertApp] Converting notebook source.ipynb to python
[NbConvertApp] Writing 1271 bytes to source.py
